# 31｜不用预制 Transformer：手写 Decoder-only GPT 与 KV Cache

本笔记从 token/位置嵌入、手写 QKV 多头注意力、Pre-Norm 残差块一直实现到 `ScratchGPT.forward` 和逐 token `forward_step`。重点不是堆出一个“大模型”，而是把三条容易写错的工程合同变成可执行证据：**未来 token 不得泄漏、全量前向与 KV Cache 的 logits 必须等价、tokenizer/config/权重必须被同一制品指纹绑定**。

> 实验边界：固定 CPU、合成小数据和受控过拟合只能验证实现链路，不能代表真实语言建模能力。

## 1. 输入、输出与验收合同

- `input_ids: [B,T]`，`valid_mask: [B,T]`；有效 token 必须是左对齐连续前缀，且每条至少一个。
- 隐状态为 `[B,T,D]`；拆头后为 `[B,H,T,D/H]`，要求 `D % H == 0`。
- full forward 同时使用 causal mask 与 key padding mask；padding query 的输出强制归零。
- 增量接口一次只接收 `[B,1]`，第 `l` 层 cache 保存 `K/V: [B,H,t,d_h]` 和 key 有效位。
- LM head 与 token embedding **共享同一 Parameter**，而不只是数值拷贝。
- 所有随机源固定；不联网、不下载数据、不依赖 `nn.MultiheadAttention/nn.Transformer`。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import copy
import hashlib
import io
import json
import math
import random

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260810
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
PAD, BOS, EOS, UNK = 0, 1, 2, 3

assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
assert len({PAD, BOS, EOS, UNK}) == 4
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. 先冻结 tokenizer 合同

生产中的模型权重依赖“字符串如何变成 id”。只校验 `state_dict` 而漏掉词表顺序，会让同一 id 悄悄换义。下面的离线 tokenizer 故意很小：按空格切分、可添加 BOS/EOS，并把 token 顺序纳入制品哈希。它不是通用分词器，只用于展示接口和版本边界。

In [ ]:
class TinyTokenizer:
    def __init__(self, tokens):
        if len(tokens) != len(set(tokens)):
            raise ValueError("词表 token 必须唯一")
        if list(tokens[:4]) != ["<pad>", "<bos>", "<eos>", "<unk>"]:
            raise ValueError("特殊 token 的 id 合同不成立")
        self.tokens = list(tokens)
        self.to_id = {token: i for i, token in enumerate(tokens)}

    def encode(self, text, add_special=True):
        ids = [self.to_id.get(piece, UNK) for piece in text.strip().split()]
        return ([BOS] + ids + [EOS]) if add_special else ids

    def decode(self, ids, skip_special=True):
        specials = {PAD, BOS, EOS}
        return " ".join(self.tokens[i] for i in ids if not (skip_special and i in specials))

    def spec(self):
        return {"kind": "whitespace-v1", "tokens": self.tokens}

TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "我", "爱", "机器", "学习",
          "模型", "需要", "数据", "检索", "图", "视觉", "文本", "安全"]
tokenizer = TinyTokenizer(TOKENS)
roundtrip = tokenizer.encode("我 爱 机器 学习")
assert roundtrip == [BOS, 4, 5, 6, 7, EOS]
assert tokenizer.decode(roundtrip) == "我 爱 机器 学习"
assert tokenizer.encode("未知词", add_special=False) == [UNK]

## 3. 缩放点积注意力与 mask 语义

对第 `h` 个头：

$$A_h=\operatorname{softmax}\left(\frac{Q_hK_h^\top}{\sqrt{d_h}}+M\right),\qquad O_h=A_hV_h.$$

`allowed_mask=True` 表示可读。full forward 的允许矩阵是

$$M_{ij}=\mathbb{1}[j\le i]\land\mathbb{1}[\text{key}_j\text{ valid}].$$

代码拒绝整行无可见 key，避免 softmax 全是负无穷时产生 NaN。padding query 虽可计算临时值，但返回前会乘 query mask 清零。

In [ ]:
def scaled_dot_product_attention(q, k, v, allowed_mask):
    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:
        raise ValueError("q/k/v 必须是 [B,H,T,d_h]")
    if k.shape != v.shape or q.shape[:2] != k.shape[:2] or q.shape[-1] != k.shape[-1]:
        raise ValueError("q/k/v 形状合同不成立")
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    allowed = torch.broadcast_to(allowed_mask, scores.shape)
    if allowed.dtype != torch.bool:
        raise TypeError("allowed_mask 必须为 bool")
    if bool((~allowed.any(-1)).any()):
        raise ValueError("存在看不到任何 key 的 query")
    scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

q = torch.tensor([[[[1.0, 0.0]]]])
k = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
v = torch.tensor([[[[3.0, 1.0], [9.0, 7.0]]]])
context, weights = scaled_dot_product_attention(q, k, v, torch.tensor([[[[True, False]]]]))
assert torch.allclose(context, v[:, :, :1])
assert torch.allclose(weights.sum(-1), torch.ones_like(weights.sum(-1)))

## 4. 手写 MHA、Pre-Norm block 与 GPT

每层先归一化再进入子层：

$$x'=x+\operatorname{MHA}(\operatorname{LN}(x)),\quad
x''=x'+\operatorname{MLP}(\operatorname{LN}(x')).$$

Pre-Norm 在深层网络中通常更容易优化。MLP 使用 $D\to4D\to D$。增量模式只为新 token 计算 Q/K/V，并将 K/V 追加到各层 cache；历史 token 的 K/V 不再重复投影。这里不用 dropout，使 train/eval 与 full/cache oracle 更容易精确比较。

In [ ]:
def validate_prefix_mask(mask):
    if mask.ndim != 2 or mask.dtype != torch.bool:
        raise ValueError("valid_mask 必须是二维 bool")
    if bool((~mask.any(1)).any()):
        raise ValueError("每条序列至少需要一个有效 token")
    if mask.shape[1] > 1 and bool((mask[:, 1:] & ~mask[:, :-1]).any()):
        raise ValueError("有效 token 必须是左对齐连续前缀")


class ManualCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        if d_model <= 0 or n_heads <= 0 or d_model % n_heads:
            raise ValueError("d_model 必须能被 n_heads 整除")
        self.d_model, self.n_heads = d_model, n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def _merge(self, x):
        B, H, T, Dh = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, H * Dh)

    def forward(self, x, valid_mask):
        B, T, D = x.shape
        if D != self.d_model or valid_mask.shape != (B, T):
            raise ValueError("attention 输入形状错误")
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))
        causal = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        allowed = causal[None, None] & valid_mask[:, None, None, :]
        context, weights = scaled_dot_product_attention(q, k, v, allowed)
        y = self.out_proj(self._merge(context))
        return y * valid_mask.unsqueeze(-1), weights

    def _validate_cache(self, cache, batch_size, expected_length, reference):
        required = {"k", "v", "valid"}
        if not isinstance(cache, dict) or set(cache) != required:
            raise ValueError("cache 字段必须严格为 k/v/valid")
        k_old, v_old, valid_old = cache["k"], cache["v"], cache["valid"]
        if not all(isinstance(value, torch.Tensor) for value in (k_old, v_old, valid_old)):
            raise ValueError("cache 字段必须是 tensor")
        expected_kv = (batch_size, self.n_heads, expected_length, self.head_dim)
        if k_old.shape != expected_kv or v_old.shape != expected_kv:
            raise ValueError(f"cache K/V shape 错误，期望 {expected_kv}")
        if valid_old.shape != (batch_size, expected_length) or valid_old.dtype != torch.bool:
            raise ValueError("cache valid shape/dtype 错误")
        if k_old.dtype != reference.dtype or v_old.dtype != reference.dtype:
            raise ValueError("cache K/V dtype 与当前请求不一致")
        if k_old.device != reference.device or v_old.device != reference.device or valid_old.device != reference.device:
            raise ValueError("cache device 与当前请求不一致")
        validate_prefix_mask(valid_old)
        return k_old, v_old, valid_old

    def forward_step(self, x_t, cache, valid_t, expected_length):
        if x_t.ndim != 3 or x_t.shape[1] != 1 or x_t.shape[-1] != self.d_model:
            raise ValueError("增量 attention 一次只接收 [B,1,D]")
        if valid_t.shape != x_t.shape[:2] or valid_t.dtype != torch.bool:
            raise ValueError("valid_t 形状或类型错误")
        if not isinstance(expected_length, int) or isinstance(expected_length, bool) or expected_length < 0:
            raise ValueError("expected_length 必须是非负整数")
        q = self._split(self.q_proj(x_t))
        k_new, v_new = self._split(self.k_proj(x_t)), self._split(self.v_proj(x_t))
        if cache is None:
            if expected_length != 0:
                raise ValueError("非零 position 不允许空 cache")
            if not bool(valid_t.all()):
                raise ValueError("每条增量序列的首 token 必须有效")
            k_all, v_all, key_valid = k_new, v_new, valid_t
        else:
            if expected_length == 0:
                raise ValueError("position=0 不允许已有 cache")
            k_old, v_old, valid_old = self._validate_cache(
                cache, x_t.shape[0], expected_length, x_t
            )
            if bool((valid_t[:, 0] & ~valid_old[:, -1]).any()):
                raise ValueError("cache 有效前缀已结束，不能再次追加有效 token")
            k_all = torch.cat([k_old, k_new], dim=2)
            v_all = torch.cat([v_old, v_new], dim=2)
            key_valid = torch.cat([valid_old, valid_t], dim=1)
        allowed = key_valid[:, None, None, :]
        context, _ = scaled_dot_product_attention(q, k_all, v_all, allowed)
        y = self.out_proj(self._merge(context)) * valid_t.unsqueeze(-1)
        return y, {"k": k_all, "v": v_all, "valid": key_valid}


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = ManualCausalSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model), nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
        )

    def forward(self, x, valid_mask):
        a, weights = self.attn(self.ln1(x), valid_mask)
        x = (x + a) * valid_mask.unsqueeze(-1)
        x = (x + self.mlp(self.ln2(x))) * valid_mask.unsqueeze(-1)
        return x, weights

    def forward_step(self, x_t, cache, valid_t, expected_length):
        a, new_cache = self.attn.forward_step(
            self.ln1(x_t), cache, valid_t, expected_length
        )
        x_t = (x_t + a) * valid_t.unsqueeze(-1)
        x_t = (x_t + self.mlp(self.ln2(x_t))) * valid_t.unsqueeze(-1)
        return x_t, new_cache


class ScratchGPT(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        if vocab_size <= 4 or max_len <= 1 or n_layers <= 0:
            raise ValueError("模型配置必须为正且词表需包含普通 token")
        self.config = dict(vocab_size=vocab_size, max_len=max_len, d_model=d_model,
                           n_heads=n_heads, n_layers=n_layers)
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def _check_ids(self, ids):
        if ids.dtype != torch.long or ids.ndim != 2:
            raise ValueError("input_ids 必须是二维 long")
        if ids.numel() and (int(ids.min()) < 0 or int(ids.max()) >= self.config["vocab_size"]):
            raise ValueError("token id 越界")

    def forward(self, input_ids, valid_mask=None):
        self._check_ids(input_ids)
        B, T = input_ids.shape
        if T > self.config["max_len"]:
            raise ValueError("序列超过 max_len")
        valid_mask = input_ids.ne(PAD) if valid_mask is None else valid_mask
        if valid_mask.shape != input_ids.shape:
            raise ValueError("mask 与 ids 形状不一致")
        validate_prefix_mask(valid_mask)
        pos = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(pos)[None]
        x = x * valid_mask.unsqueeze(-1)
        all_weights = []
        for block in self.blocks:
            x, weights = block(x, valid_mask)
            all_weights.append(weights)
        x = self.final_norm(x) * valid_mask.unsqueeze(-1)
        return self.lm_head(x), all_weights

    def _validate_cache_envelope(self, caches, batch_size, position, request_id, valid_t, device):
        if caches is None:
            if position != 0:
                raise ValueError("空 cache 时 position 必须为 0")
            if not bool(valid_t.all()):
                raise ValueError("每条请求的首 token 必须有效")
            return [None] * len(self.blocks), torch.ones(batch_size, dtype=torch.bool, device=device)
        required = {"schema", "request_id", "batch_size", "next_position", "prefix_open", "layers"}
        if not isinstance(caches, dict) or set(caches) != required:
            raise ValueError("cache envelope 字段不完整")
        if caches["schema"] != "scratch-gpt-kv-v1":
            raise ValueError("cache schema 不支持")
        if caches["request_id"] != request_id:
            raise ValueError("cache 不属于当前请求")
        if caches["batch_size"] != batch_size:
            raise ValueError("cache batch size 与当前请求不一致")
        if caches["next_position"] != position:
            raise ValueError("position 必须严格等于 cache length")
        prefix_open = caches["prefix_open"]
        if not isinstance(prefix_open, torch.Tensor) or prefix_open.shape != (batch_size,):
            raise ValueError("cache prefix_open shape 错误")
        if prefix_open.dtype != torch.bool or prefix_open.device != device:
            raise ValueError("cache prefix_open dtype/device 错误")
        layers = caches["layers"]
        if not isinstance(layers, list) or len(layers) != len(self.blocks):
            raise ValueError("cache 层数与模型不一致")
        if bool((valid_t[:, 0] & ~prefix_open).any()):
            raise ValueError("有效前缀结束后不能再次追加普通 token")
        return layers, prefix_open

    def forward_step(self, token_id, caches=None, position=0, valid_t=None, request_id=None):
        self._check_ids(token_id)
        if token_id.shape[1] != 1:
            raise ValueError("forward_step 一次只接收一个 token")
        if not isinstance(position, int) or isinstance(position, bool):
            raise ValueError("position 必须是整数")
        if not 0 <= position < self.config["max_len"]:
            raise ValueError("position 越界")
        if not isinstance(request_id, str) or not request_id:
            raise ValueError("request_id 必须是非空字符串")
        B = token_id.shape[0]
        valid_t = token_id.ne(PAD) if valid_t is None else valid_t
        if valid_t.shape != (B, 1) or valid_t.dtype != torch.bool or valid_t.device != token_id.device:
            raise ValueError("valid_t shape/dtype/device 合同错误")
        if not torch.equal(valid_t, token_id.ne(PAD)):
            raise ValueError("valid_t 必须与 token_id 的 PAD 语义一致")
        layer_caches, prefix_open = self._validate_cache_envelope(
            caches, B, position, request_id, valid_t, token_id.device
        )
        pos = torch.tensor([position], device=token_id.device)
        x = (self.token_embedding(token_id) + self.position_embedding(pos)[None]) * valid_t.unsqueeze(-1)
        new_layers = []
        for block, layer_cache in zip(self.blocks, layer_caches):
            x, new_cache = block.forward_step(x, layer_cache, valid_t, expected_length=position)
            new_layers.append(new_cache)
        reference_valid = new_layers[0]["valid"]
        if any(not torch.equal(layer["valid"], reference_valid) for layer in new_layers[1:]):
            raise RuntimeError("各层 cache valid 状态不一致")
        x = self.final_norm(x) * valid_t.unsqueeze(-1)
        new_envelope = {
            "schema": "scratch-gpt-kv-v1",
            "request_id": request_id,
            "batch_size": B,
            "next_position": position + 1,
            "prefix_open": prefix_open & valid_t[:, 0],
            "layers": new_layers,
        }
        return self.lm_head(x), new_envelope

In [ ]:
gpt = ScratchGPT(len(TOKENS), max_len=12, d_model=24, n_heads=4, n_layers=2).to(DEVICE)
sample_ids = torch.tensor([[BOS, 4, 5, EOS, PAD], [BOS, 8, 9, 10, EOS]])
sample_mask = sample_ids.ne(PAD)
sample_logits, sample_weights = gpt(sample_ids, sample_mask)

assert sample_logits.shape == (2, 5, len(TOKENS))
assert len(sample_weights) == 2 and sample_weights[0].shape == (2, 4, 5, 5)
assert gpt.lm_head.weight is gpt.token_embedding.weight
assert torch.count_nonzero(sample_logits[0, 4]) == 0
assert sum(p.numel() for p in gpt.parameters()) < 50_000

try:
    gpt(torch.tensor([[BOS, PAD, 4]]), torch.tensor([[True, False, True]]))
    raise AssertionError("非前缀 mask 应被拒绝")
except ValueError as exc:
    assert "左对齐" in str(exc)

## 5. 三个关键 oracle：未来隔离、KV Cache 等价与 cache 状态机

1. **未来 token 干预**：只改位置 `j`，所有 `i<j` 的 logits 必须逐元素不变。
2. **缓存等价**：eval 模式从位置 0 逐步喂入 token，拼接出的 logits 应与一次性 full forward 相同；还必须覆盖同 batch 中不同有效长度。
3. **状态机 fail closed**：`position` 必须等于 cache 的下一位置；每层 K/V 必须与当前 batch、head、长度、dtype、device 对齐；request id 必须一致；某行一旦进入 padding，就不能再次追加普通 token。

cache 使用 envelope 保存请求、batch、下一位置、前缀是否仍开放和逐层 K/V。它不是可跨请求复制的普通字典。在长度为 `T` 的生成中，full 重算累计约为 $O(T^3D)$，KV Cache 约为 $O(T^2D)$，代价是每层 $O(BTD)$ cache 内存。

In [ ]:
gpt.eval()
probe = torch.tensor([[BOS, 4, 5, 6, 7, EOS], [BOS, 8, 9, 10, 11, EOS]])
probe_mask = torch.ones_like(probe, dtype=torch.bool)
changed = probe.clone()
changed[:, 4] = torch.tensor([12, 13])

with torch.no_grad():
    full_logits, _ = gpt(probe, probe_mask)
    changed_logits, _ = gpt(changed, probe_mask)
assert torch.equal(full_logits[:, :4], changed_logits[:, :4])
assert not torch.allclose(full_logits[:, 4], changed_logits[:, 4])

caches, step_logits = None, []
with torch.no_grad():
    for position in range(probe.shape[1]):
        logits_t, caches = gpt.forward_step(
            probe[:, position:position + 1], caches, position,
            valid_t=probe_mask[:, position:position + 1], request_id="oracle-full"
        )
        step_logits.append(logits_t)
cached_logits = torch.cat(step_logits, dim=1)
max_cache_error = (cached_logits - full_logits).abs().max().item()
assert torch.allclose(cached_logits, full_logits, atol=7e-6, rtol=1e-5), max_cache_error
assert caches["next_position"] == probe.shape[1]
assert all(layer["k"].shape[2] == probe.shape[1] for layer in caches["layers"])
assert all(layer["valid"].all() for layer in caches["layers"])

# 同一个 batch 中第一行有尾部 padding，full 与 cache 仍需一致。
padded = torch.tensor([[BOS, 4, 5, EOS, PAD, PAD], [BOS, 8, 9, 10, 11, EOS]])
padded_mask = padded.ne(PAD)
with torch.no_grad():
    padded_full = gpt(padded, padded_mask)[0]
    padded_cache, padded_steps = None, []
    for position in range(padded.shape[1]):
        logits_t, padded_cache = gpt.forward_step(
            padded[:, position:position + 1], padded_cache, position,
            valid_t=padded_mask[:, position:position + 1], request_id="oracle-padded"
        )
        padded_steps.append(logits_t)
padded_cached = torch.cat(padded_steps, dim=1)
padded_error = float((padded_cached - padded_full).abs().max())
assert torch.allclose(padded_cached, padded_full, atol=7e-6, rtol=1e-5), padded_error
assert padded_cache["prefix_open"].tolist() == [False, True]

# position、request、shape 与有效前缀反例必须在接口层被拒绝。
with torch.no_grad():
    _, one_step_cache = gpt.forward_step(
        torch.tensor([[BOS]]), None, 0, request_id="request-a"
    )

def expect_cache_error(action, phrase):
    try:
        action()
        raise AssertionError("非法 cache 必须 fail closed")
    except ValueError as exc:
        assert phrase in str(exc), str(exc)

expect_cache_error(
    lambda: gpt.forward_step(torch.tensor([[4]]), one_step_cache, 3, request_id="request-a"),
    "position",
)
expect_cache_error(
    lambda: gpt.forward_step(torch.tensor([[4]]), one_step_cache, 1, request_id="request-b"),
    "请求",
)
bad_shape_cache = copy.deepcopy(one_step_cache)
bad_shape_cache["layers"][0]["k"] = bad_shape_cache["layers"][0]["k"][:, :-1]
expect_cache_error(
    lambda: gpt.forward_step(torch.tensor([[4]]), bad_shape_cache, 1, request_id="request-a"),
    "shape",
)
with torch.no_grad():
    _, gap_cache = gpt.forward_step(torch.tensor([[BOS]]), None, 0, request_id="gap")
    _, gap_cache = gpt.forward_step(torch.tensor([[PAD]]), gap_cache, 1, request_id="gap")
expect_cache_error(
    lambda: gpt.forward_step(torch.tensor([[4]]), gap_cache, 2, request_id="gap"),
    "前缀",
)
expect_cache_error(
    lambda: gpt.forward_step(
        torch.tensor([[PAD]]), None, 0,
        valid_t=torch.ones(1, 1, dtype=torch.bool), request_id="pad-semantic"
    ),
    "PAD",
)
print({"full_vs_cache_max_abs": max_cache_error,
       "padded_full_vs_cache_max_abs": padded_error,
       "cache_rejections": ["position", "request", "shape", "valid_hole", "pad_semantic"]})

## 6. Causal LM 目标与受控过拟合

teacher forcing 的第 `t` 个 logits 预测第 `t+1` 个 token：

$$\mathcal L=-\frac{1}{N}\sum_{b,t:\,m_{b,t+1}=1}\log p(x_{b,t+1}\mid x_{b,\le t}).$$

padding 位置不是一个“PAD 类别预测任务”，因此监督标签改为 `ignore_index=-100`。下面同时以布尔选取手算一次 loss，防止 shift 或 mask 错一位。训练只要求记住固定合成序列，是实现冒烟测试，不是泛化基准。

In [ ]:
def make_toy_sequences():
    rows = []
    for i in range(12):
        length = 6 + (i % 3)
        regular = [4 + ((i + j) % (len(TOKENS) - 4)) for j in range(length - 2)]
        row = [BOS] + regular + [EOS]
        rows.append(row + [PAD] * (9 - len(row)))
    return torch.tensor(rows, dtype=torch.long)


def causal_lm_loss(logits, input_ids, valid_mask):
    if input_ids.ndim != 2 or input_ids.dtype != torch.long:
        raise ValueError("input_ids 必须是二维 long")
    if logits.shape[:2] != input_ids.shape or valid_mask.shape != input_ids.shape:
        raise ValueError("loss 输入形状不一致")
    if valid_mask.dtype != torch.bool or logits.shape[-1] <= int(input_ids.max()):
        raise ValueError("loss mask dtype 或 vocab 维错误")
    if not torch.equal(valid_mask, input_ids.ne(PAD)):
        raise ValueError("loss valid_mask 必须与 PAD 位置一致")
    validate_prefix_mask(valid_mask)
    supervised = valid_mask[:, 1:]
    if input_ids.shape[1] < 2 or not bool(supervised.any()):
        raise ValueError("当前 batch 没有有效 next-token label")
    targets = input_ids[:, 1:].clone()
    targets[~supervised] = -100
    return F.cross_entropy(
        logits[:, :-1].reshape(-1, logits.shape[-1]), targets.reshape(-1), ignore_index=-100
    )


toy_ids = make_toy_sequences().to(DEVICE)
toy_mask = toy_ids.ne(PAD)
test_logits, _ = gpt(toy_ids, toy_mask)
loss_api = causal_lm_loss(test_logits, toy_ids, toy_mask)
selected_logits = test_logits[:, :-1][toy_mask[:, 1:]]
selected_targets = toy_ids[:, 1:][toy_mask[:, 1:]]
loss_manual = F.cross_entropy(selected_logits, selected_targets)
assert torch.allclose(loss_api, loss_manual)
assert int(toy_mask[:, 1:].sum()) == selected_targets.numel()

single_token = torch.tensor([[BOS, PAD]])
try:
    causal_lm_loss(gpt(single_token, single_token.ne(PAD))[0], single_token, single_token.ne(PAD))
    raise AssertionError("全 ignore label 必须被拒绝")
except ValueError as exc:
    assert "next-token" in str(exc)

In [ ]:
torch.manual_seed(SEED + 1)
gpt = ScratchGPT(len(TOKENS), max_len=12, d_model=24, n_heads=4, n_layers=2).to(DEVICE)
optimizer = torch.optim.AdamW(gpt.parameters(), lr=0.025, weight_decay=0.0)

gpt.train()
with torch.no_grad():
    initial_loss = float(causal_lm_loss(gpt(toy_ids, toy_mask)[0], toy_ids, toy_mask))
for step in range(100):
    optimizer.zero_grad(set_to_none=True)
    loss = causal_lm_loss(gpt(toy_ids, toy_mask)[0], toy_ids, toy_mask)
    loss.backward()
    if step == 0:
        grad_norm = torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        assert torch.isfinite(grad_norm) and float(grad_norm) > 0
    else:
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
    optimizer.step()

gpt.eval()
with torch.no_grad():
    final_loss = float(causal_lm_loss(gpt(toy_ids, toy_mask)[0], toy_ids, toy_mask))
assert final_loss < initial_loss * 0.35, (initial_loss, final_loss)
assert torch.isfinite(torch.tensor(final_loss))
print({"initial_loss": round(initial_loss, 4), "final_loss": round(final_loss, 4)})

## 7. top-k / temperature 增量生成

temperature 对 logits 做 $z/\tau$；top-k 只在最高的 k 个候选中采样。生成 prompt 必须是无 PAD 的有效前缀，负 token 预算直接拒绝。PAD 永远从候选中屏蔽，因此不会出现“先 PAD、后普通 token”的非法 cache；EOS 立即停止。

循环只在还需要下一次采样时，才把刚生成 token 送入 `forward_step`，避免生成完最后一个 token 后做一次无用 cache 更新。相同请求级 generator seed 必须得到相同序列；真实服务还需要停止词、敏感内容策略、重复惩罚和不可伪造的请求身份。

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens, temperature=1.0, top_k=5, seed=0):
    if temperature <= 0:
        raise ValueError("temperature 必须为正")
    if not isinstance(max_new_tokens, int) or isinstance(max_new_tokens, bool) or max_new_tokens < 0:
        raise ValueError("max_new_tokens 必须是非负整数")
    if not 1 <= top_k <= model.config["vocab_size"] - 1:
        raise ValueError("top_k 越界；PAD 不属于候选")
    if prompt.ndim != 2 or prompt.shape[0] != 1 or prompt.shape[1] == 0:
        raise ValueError("示例生成器只接收非空单样本 [1,T]")
    model._check_ids(prompt)
    prompt_mask = prompt.ne(PAD)
    validate_prefix_mask(prompt_mask)
    if not bool(prompt_mask.all()):
        raise ValueError("生成 prompt 不允许包含 PAD 或 mask hole")
    if prompt.device != next(model.parameters()).device:
        raise ValueError("prompt 与模型必须位于同一 device")
    if prompt.shape[1] + max_new_tokens > model.config["max_len"]:
        raise ValueError("生成长度超过位置表")
    generated = prompt.clone()
    if max_new_tokens == 0 or int(prompt[0, -1]) == EOS:
        return generated
    model.eval()
    request_id = f"generation:{int(seed)}:{prompt.shape[1]}"
    caches, last_logits = None, None
    for position in range(prompt.shape[1]):
        last_logits, caches = model.forward_step(
            prompt[:, position:position + 1], caches, position,
            valid_t=torch.ones(1, 1, dtype=torch.bool, device=prompt.device),
            request_id=request_id,
        )
    rng = torch.Generator(device=prompt.device).manual_seed(int(seed))
    for step in range(max_new_tokens):
        scores = (last_logits[:, -1] / temperature).clone()
        scores[:, PAD] = float("-inf")
        top_values, top_indices = torch.topk(scores, top_k, dim=-1)
        probs = torch.softmax(top_values, dim=-1)
        choice = torch.multinomial(probs, 1, generator=rng)
        next_id = top_indices.gather(-1, choice)
        if int(next_id.item()) == PAD:
            raise RuntimeError("PAD 已从生成候选中排除")
        generated = torch.cat([generated, next_id], dim=1)
        if int(next_id.item()) == EOS or step == max_new_tokens - 1:
            break
        position = prompt.shape[1] + step
        last_logits, caches = model.forward_step(
            next_id, caches, position,
            valid_t=torch.ones_like(next_id, dtype=torch.bool), request_id=request_id
        )
    return generated


prompt = torch.tensor([[BOS, 4]])
generated_a = generate(gpt, prompt, 5, temperature=0.8, top_k=4, seed=77)
generated_b = generate(gpt, prompt, 5, temperature=0.8, top_k=4, seed=77)
assert torch.equal(generated_a, generated_b)
assert torch.equal(generated_a[:, :2], prompt)
assert generated_a.shape[1] <= 7 and not bool(generated_a.eq(PAD).any())
assert torch.equal(generate(gpt, prompt, 0), prompt)

for invalid_prompt, budget, phrase in [
    (prompt, -1, "非负"),
    (torch.tensor([[BOS, PAD, 4]]), 1, "前缀"),
    (torch.tensor([[BOS, 4, PAD]]), 1, "PAD"),
]:
    try:
        generate(gpt, invalid_prompt, budget, seed=1)
        raise AssertionError("非法生成请求必须被拒绝")
    except ValueError as exc:
        assert phrase in str(exc), str(exc)
print({"ids": generated_a.tolist()[0], "text": tokenizer.decode(generated_a.tolist()[0]),
       "generation_rejections": ["negative_budget", "mask_hole", "trailing_pad"]})

## 8. 发布者 registry 才是信任锚

package 内部的 SHA-256 只能发现偶然损坏：攻击者能同时替换模型、manifest 和内部哈希。这里先由“发布者侧”把 `(artifact_id, version) -> immutable manifest digest` 放入只读 registry；loader 必须先命中该外部锚，再验证原始 state bytes，最后按 tensor key、dtype、shape、bytes 重算语义权重哈希。

manifest 原子绑定完整 tokenizer token 顺序、无分类 label 的显式声明、训练数据与 split、预处理和训练 recipe。Notebook 中的 `MappingProxyType` 只是离线模拟；生产 registry 应位于受权限控制的数据库或签名透明日志，调用方不能写。

In [ ]:
from types import MappingProxyType

def canonical_hash(obj):
    raw = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(raw).hexdigest()


def tensor_state_hash(state_dict):
    digest = hashlib.sha256()
    for key in sorted(state_dict):
        tensor = state_dict[key]
        if not isinstance(tensor, torch.Tensor):
            raise TypeError("state_dict 只能包含 tensor")
        value = tensor.detach().cpu().contiguous()
        descriptor = {"key": key, "dtype": str(value.dtype), "shape": list(value.shape)}
        digest.update(json.dumps(descriptor, sort_keys=True, separators=(",", ":")).encode())
        digest.update(value.numpy().tobytes(order="C"))
    return digest.hexdigest()


def serialize_state(state_dict):
    buffer = io.BytesIO()
    torch.save(state_dict, buffer)
    return buffer.getvalue()


def validate_gpt_manifest(manifest):
    required = {"schema", "artifact_id", "version", "subject", "config", "tokenizer",
                "label_map", "preprocess", "training_snapshot", "state_bytes_sha256",
                "state_tensor_sha256"}
    if not isinstance(manifest, dict) or set(manifest) != required:
        raise ValueError("manifest schema 字段不完整")
    if manifest["schema"] != "scratch-gpt-v2" or not manifest["artifact_id"] or not manifest["version"]:
        raise ValueError("manifest 身份字段错误")
    config = manifest["config"]
    if set(config) != {"vocab_size", "max_len", "d_model", "n_heads", "n_layers"}:
        raise ValueError("GPT config 字段错误")
    tok_spec = manifest["tokenizer"]
    if set(tok_spec) != {"kind", "tokens"} or tok_spec["kind"] != "whitespace-v1":
        raise ValueError("tokenizer spec 错误")
    tok = TinyTokenizer(tok_spec["tokens"])
    if config["vocab_size"] != len(tok.tokens):
        raise ValueError("vocab_size 与 tokenizer token 数不一致")
    if manifest["label_map"] is not None:
        raise ValueError("causal LM 不应伪造分类 label map")
    preprocess = manifest["preprocess"]
    expected_preprocess = {"splitter": "whitespace-v1", "padding": "right",
                           "pad_id": PAD, "bos_id": BOS, "eos_id": EOS,
                           "unk_id": UNK, "max_len": config["max_len"]}
    if preprocess != expected_preprocess:
        raise ValueError("预处理快照与模型合同不一致")
    snapshot = manifest["training_snapshot"]
    if snapshot.get("tokenizer_sha256") != canonical_hash(tok_spec):
        raise ValueError("训练快照 tokenizer 指纹不一致")
    dataset = snapshot.get("dataset", {})
    rows, split = dataset.get("rows"), dataset.get("split")
    if not isinstance(rows, list) or not rows or set(split or {}) != {"train", "validation", "test"}:
        raise ValueError("训练数据或 split 快照错误")
    all_indices = split["train"] + split["validation"] + split["test"]
    if len(all_indices) != len(set(all_indices)) or sorted(all_indices) != list(range(len(rows))):
        raise ValueError("训练 split 必须互斥且覆盖数据快照")
    for row in rows:
        if not isinstance(row, list) or not 2 <= len(row) <= config["max_len"]:
            raise ValueError("训练行长度错误")
        if any(not isinstance(token, int) or token < 0 or token >= config["vocab_size"] for token in row):
            raise ValueError("训练 token id 越界")
        valid = [token != PAD for token in row]
        if any(valid[i] and not valid[i - 1] for i in range(1, len(valid))):
            raise ValueError("训练数据包含 padding hole")
    recipe = snapshot.get("recipe", {})
    expected_recipe = {"objective": "causal-next-token", "optimizer": "AdamW",
                       "steps": 100, "lr": 0.025, "weight_decay": 0.0,
                       "clip_grad_norm": 1.0, "seed": SEED + 1,
                       "ignore_index": -100, "purpose": "controlled-overfit"}
    if recipe != expected_recipe:
        raise ValueError("训练 recipe 快照不一致")
    return tok


def build_artifact(model, tok, subject, artifact_id, version, training_snapshot):
    state_dict = model.state_dict()
    state_bytes = serialize_state(state_dict)
    manifest = {
        "schema": "scratch-gpt-v2", "artifact_id": artifact_id, "version": version,
        "subject": subject, "config": copy.deepcopy(model.config),
        "tokenizer": copy.deepcopy(tok.spec()), "label_map": None,
        "preprocess": {"splitter": "whitespace-v1", "padding": "right", "pad_id": PAD,
                       "bos_id": BOS, "eos_id": EOS, "unk_id": UNK,
                       "max_len": model.config["max_len"]},
        "training_snapshot": copy.deepcopy(training_snapshot),
        "state_bytes_sha256": hashlib.sha256(state_bytes).hexdigest(),
        "state_tensor_sha256": tensor_state_hash(state_dict),
    }
    validate_gpt_manifest(manifest)
    return {"manifest": manifest, "manifest_sha256": canonical_hash(manifest),
            "state_bytes": state_bytes}


training_snapshot31 = {
    "dataset": {"name": "toy-causal-sequences-v1", "rows": toy_ids.cpu().tolist(),
                "split": {"train": list(range(len(toy_ids))), "validation": [], "test": []}},
    "tokenizer_sha256": canonical_hash(tokenizer.spec()),
    "recipe": {"objective": "causal-next-token", "optimizer": "AdamW", "steps": 100,
               "lr": 0.025, "weight_decay": 0.0, "clip_grad_norm": 1.0,
               "seed": SEED + 1, "ignore_index": -100, "purpose": "controlled-overfit"},
}
artifact = build_artifact(
    gpt, tokenizer, subject="nlp-lab/gpt-demo", artifact_id="scratch-gpt-demo", version="1.0.0",
    training_snapshot=training_snapshot31,
)
# 这一步模拟发布系统写入的外部、只读信任锚；请求方只能提交 package，不能改 registry。
PUBLISHER_REGISTRY31 = MappingProxyType({
    ("scratch-gpt-demo", "1.0.0"): artifact["manifest_sha256"]
})


def trusted_load(package, expected_subject):
    if not isinstance(package, dict) or set(package) != {"manifest", "manifest_sha256", "state_bytes"}:
        raise ValueError("package 字段错误")
    manifest = package["manifest"]
    key = (manifest.get("artifact_id"), manifest.get("version"))
    expected_digest = PUBLISHER_REGISTRY31.get(key)
    if expected_digest is None:
        raise PermissionError("artifact id/version 未由发布者注册")
    computed_manifest_digest = canonical_hash(manifest)
    if package["manifest_sha256"] != computed_manifest_digest:
        raise ValueError("package 内 manifest hash 不一致")
    if computed_manifest_digest != expected_digest:
        raise PermissionError("package 内容不匹配发布者 registry")
    if manifest.get("subject") != expected_subject:
        raise PermissionError("业务主体不匹配")
    if hashlib.sha256(package["state_bytes"]).hexdigest() != manifest["state_bytes_sha256"]:
        raise ValueError("原始权重字节指纹不匹配")
    tok = validate_gpt_manifest(manifest)
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)
    if tensor_state_hash(state) != manifest["state_tensor_sha256"]:
        raise ValueError("tensor key/dtype/shape/bytes 指纹不匹配")
    model = ScratchGPT(**manifest["config"])
    model.load_state_dict(state, strict=True)
    return model.eval(), tok


loaded_gpt, loaded_tokenizer = trusted_load(artifact, "nlp-lab/gpt-demo")
with torch.no_grad():
    original = gpt(probe, probe_mask)[0]
    restored = loaded_gpt(probe, probe_mask)[0]
assert torch.equal(original, restored)
assert loaded_tokenizer.tokens == tokenizer.tokens
assert gpt.lm_head.weight is gpt.token_embedding.weight

# 仅修改内容并重算 package 内 hash，仍不能改变发布者 registry 中的预期 digest。
resigned_config = copy.deepcopy(artifact)
resigned_config["manifest"]["config"]["max_len"] += 1
resigned_config["manifest_sha256"] = canonical_hash(resigned_config["manifest"])
forged_model = ScratchGPT(**gpt.config)
with torch.no_grad():
    for parameter in forged_model.parameters():
        parameter.zero_()
fully_resigned = build_artifact(
    forged_model, tokenizer, "nlp-lab/gpt-demo", "scratch-gpt-demo", "1.0.0", training_snapshot31
)
for candidate in (resigned_config, fully_resigned):
    try:
        trusted_load(candidate, "nlp-lab/gpt-demo")
        raise AssertionError("整体重签伪造必须被 registry 拒绝")
    except PermissionError as exc:
        assert "registry" in str(exc)

mismatched_vocab_model = ScratchGPT(len(TOKENS) + 1, max_len=12, d_model=24, n_heads=4, n_layers=2)
try:
    build_artifact(
        mismatched_vocab_model, tokenizer, "nlp-lab/gpt-demo",
        "mismatch", "1.0.0", training_snapshot31
    )
    raise AssertionError("vocab_size/tokenizer mismatch 必须在发布前拒绝")
except ValueError as exc:
    assert "vocab_size" in str(exc)

try:
    trusted_load(artifact, "another-tenant")
    raise AssertionError("跨主体加载不应通过")
except PermissionError:
    pass
print({"artifact_registry": dict(PUBLISHER_REGISTRY31),
       "tensor_state_sha256_prefix": artifact["manifest"]["state_tensor_sha256"][:12],
       "resigned_forgery_rejected": True, "snapshot_rows": len(training_snapshot31["dataset"]["rows"])})

## 9. 复杂度、失败模式与生产边界

- 单层 full self-attention 时间/注意力矩阵空间分别约为 $O(BT^2D)$ / $O(BHT^2)$；MLP 时间约为 $O(BTD^2)$。
- KV Cache 降低重复计算，却随 batch、层数、上下文长度线性占显存；生产需分页 cache、配额、淘汰和请求隔离。
- 常见错误：cache 的位置 embedding 从 0 重启；只缓存最后一层；把 `True` 同时解释为“允许”和“屏蔽”；在训练态比较 dropout；权重共享发生在创建 optimizer 之后。
- padding 不是普通词。若右侧 PAD 参与 loss 或成为可见 key，模型会学习数据管道伪特征。
- 本实现没有 RoPE、FlashAttention、混合精度、分布式训练和连续批处理；这些是部署扩展，不应混进最小正确性 oracle。
- SHA-256 只证明字节未变，不能证明发布者可信；真实性需要签名和可信密钥。

## 10. 原始论文与官方资料

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762)：缩放点积、多头注意力与位置表示。
- Radford et al., [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)：decoder-style 生成式预训练。
- PyTorch 官方文档：[Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)、[Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)、[cross_entropy](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)。

复现论文时应记录数据版本、tokenizer、超参数和评估脚本；本笔记引用结构思想，不声称复现论文规模或指标。